# Catalog Enrichment Pipeline Demo

## End-to-End Product Data Enrichment from Raw Supplier Feeds

This notebook demonstrates the enrichment pipeline described in the [GenAI Product Catalog Architecture](../architectures/genai-product-catalog/architecture.md). We take a raw, incomplete supplier feed and produce fully enriched product records with:

1. Extracted attributes from unstructured text
2. Generated product descriptions for multiple channels
3. Taxonomy classification
4. Quality scoring and validation

**Prerequisites:** `boto3`, `pandas`, `json`

**Note:** Uses synthetic product data. No real supplier or customer data included.

## 1. Simulate Raw Supplier Feed

In production, this data arrives via EDI (X12 832), XML (GS1 GDSN), or flat files from suppliers. It is always incomplete. Suppliers provide what they have, which is rarely what retailers need.

In [ ]:
import pandas as pd
import json

# Simulated raw supplier feed — intentionally incomplete
raw_feed = [
    {
        "supplier_sku": "NP-OAT-032-ORG",
        "supplier_name": "Pacific Northwest Organics",
        "product_name": "Organic Oat Milk Original",
        "upc": "058449000123",
        "brand": "Nature's Path",
        "size": "32 fl oz",
        "case_pack": 12,
        "wholesale_cost": 3.49,
        "description": "Our creamy oat milk made from organic oats. Great in coffee, cereal, or on its own. No artificial flavors.",
        # Missing: category, allergens, dietary_claims, storage_temp,
        # shelf_life, flavor, ingredients_list, certifications
    },
    {
        "supplier_sku": "GV-CHIPS-FAM-BBQ",
        "supplier_name": "Great Valley Snacks",
        "product_name": "Family Size BBQ Kettle Chips",
        "upc": "072251003456",
        "brand": "Crunchy Valley",
        "size": "16 oz",
        "case_pack": 8,
        "wholesale_cost": 4.29,
        "description": "Thick-cut kettle chips with bold BBQ seasoning. Cooked in small batches.",
        # Missing: category, allergens, dietary_claims, storage_temp,
        # shelf_life, flavor, cooking_method, texture
    },
    {
        "supplier_sku": "BW-SOAP-LVD-3PK",
        "supplier_name": "Botanica Wellness",
        "product_name": "Lavender Calm Hand Soap 3-Pack",
        "upc": "081592007890",
        "brand": "Botanica",
        "size": "10 fl oz each",
        "case_pack": 6,
        "wholesale_cost": 8.99,
        "description": "Gentle hand soap infused with lavender essential oil. Sulfate-free formula.",
        # Missing: category, scent_family, skin_type, certifications,
        # packaging_material, cruelty_free_status
    }
]

df_raw = pd.DataFrame(raw_feed)
print(f"Raw feed: {len(df_raw)} products")
print(f"Available fields: {list(df_raw.columns)}")
print(f"\nNotice what's MISSING: category, allergens, dietary claims, storage,")
print(f"shelf life, certifications — all required for retail catalog compliance.")
print(f"\n--- Sample Product ---")
print(json.dumps(raw_feed[0], indent=2))

## 2. Define Target Taxonomy

The enterprise taxonomy defines what a complete product record looks like. Every attribute has a type, allowed values (for enums), and whether it is required for catalog publication.

In production, this is maintained in the PIM system and exported as the enrichment pipeline's validation schema.

In [ ]:
# Enterprise product taxonomy (simplified)
taxonomy = {
    "category_tree": {
        "Beverages": ["Dairy Alternatives", "Juice", "Water", "Carbonated", "Coffee & Tea"],
        "Snacks": ["Chips & Crisps", "Crackers", "Nuts & Seeds", "Bars", "Popcorn"],
        "Health & Beauty": ["Hand & Body Care", "Hair Care", "Oral Care", "Skin Care"]
    },
    "required_attributes": {
        "all_categories": [
            {"name": "category_l1", "type": "enum"},
            {"name": "category_l2", "type": "enum"},
            {"name": "storage_temp", "type": "enum", "values": ["Ambient", "Refrigerated", "Frozen"]},
            {"name": "shelf_life_days", "type": "integer", "range": [1, 1095]}
        ],
        "food_and_beverage": [
            {"name": "allergens", "type": "multi_enum", 
             "values": ["Milk", "Soy", "Tree Nuts", "Wheat", "Eggs", "Peanuts", "Fish", "Shellfish", "Sesame", "None"]},
            {"name": "dietary_claims", "type": "multi_enum",
             "values": ["Vegan", "Vegetarian", "Gluten-Free", "Non-GMO", "Organic", "Kosher", "Halal", "Keto-Friendly"]},
            {"name": "ingredients_summary", "type": "text"}
        ],
        "health_and_beauty": [
            {"name": "cruelty_free", "type": "boolean"},
            {"name": "paraben_free", "type": "boolean"},
            {"name": "key_ingredients", "type": "text"}
        ]
    },
    "description_channels": [
        {"channel": "ecommerce", "max_words": 300, "tone": "consumer-friendly"},
        {"channel": "marketplace", "max_words": 150, "tone": "seo-optimized"},
        {"channel": "b2b_catalog", "max_words": 100, "tone": "technical"},
        {"channel": "shelf_tag", "max_words": 25, "tone": "minimal"}
    ]
}

print("Taxonomy loaded.")
print(f"  Category L1 options: {list(taxonomy['category_tree'].keys())}")
print(f"  Required attributes (all): {len(taxonomy['required_attributes']['all_categories'])}")
print(f"  Required attributes (food): {len(taxonomy['required_attributes']['food_and_beverage'])} additional")
print(f"  Description channels: {len(taxonomy['description_channels'])}")

## 3. Attribute Enrichment via Foundation Model

We send each product to Bedrock with:
- The known attributes from the supplier feed
- The target attribute schema from our taxonomy
- Instructions to extract or infer attributes, returning null when confidence is low

The prompt is structured to produce constrained JSON output that matches our taxonomy's allowed values.

In [ ]:
def build_enrichment_prompt(product, taxonomy):
    """Build the attribute enrichment prompt for a single product."""
    
    # Determine which attribute set applies
    food_keywords = ["milk", "chips", "snack", "beverage", "drink", "food", "cereal"]
    is_food = any(kw in product['product_name'].lower() or kw in product['description'].lower() 
                  for kw in food_keywords)
    
    target_attrs = taxonomy['required_attributes']['all_categories'].copy()
    if is_food:
        target_attrs.extend(taxonomy['required_attributes']['food_and_beverage'])
    else:
        target_attrs.extend(taxonomy['required_attributes']['health_and_beauty'])
    
    attr_spec = json.dumps(target_attrs, indent=2)
    categories = json.dumps(taxonomy['category_tree'], indent=2)
    
    prompt = f"""You are a product data specialist for a retail enterprise. Extract or infer 
product attributes from the available information.

## Product Information (from supplier)
- Name: {product['product_name']}
- Brand: {product['brand']}
- Size: {product['size']}
- Description: {product['description']}

## Category Tree
{categories}

## Target Attributes to Extract
{attr_spec}

## Rules
1. For enum types, only use values from the allowed list.
2. If you cannot determine an attribute with reasonable confidence, return null.
3. For allergens, only assert allergens explicitly mentioned or clearly implied by 
   the product type. "Oat milk" implies potential wheat/gluten cross-contamination 
   but NOT confirmed allergen presence — return what is explicitly stated only.
4. For dietary claims, only assert claims supported by the description.

Respond with a JSON object mapping attribute_name to extracted value. Include a 
"_confidence" object mapping each attribute_name to a confidence score (0.0-1.0)."""
    
    return prompt, is_food

# Build prompt for first product
prompt, is_food = build_enrichment_prompt(raw_feed[0], taxonomy)
print(f"Product: {raw_feed[0]['product_name']}")
print(f"Detected as food/beverage: {is_food}")
print(f"Prompt length: {len(prompt)} characters")
print(f"\n--- Prompt excerpt (first 400 chars) ---")
print(prompt[:400])

In [ ]:
# Simulated Bedrock responses (representative of actual model output)
enrichment_results = [
    {
        "product": "Organic Oat Milk Original",
        "attributes": {
            "category_l1": "Beverages",
            "category_l2": "Dairy Alternatives",
            "storage_temp": "Ambient",
            "shelf_life_days": 270,
            "allergens": ["None"],
            "dietary_claims": ["Vegan", "Organic"],
            "ingredients_summary": "Organic oats, filtered water, organic sunflower oil, sea salt, gellan gum"
        },
        "_confidence": {
            "category_l1": 0.99,
            "category_l2": 0.97,
            "storage_temp": 0.75,
            "shelf_life_days": 0.60,
            "allergens": 0.70,
            "dietary_claims": 0.88,
            "ingredients_summary": 0.45
        }
    },
    {
        "product": "Family Size BBQ Kettle Chips",
        "attributes": {
            "category_l1": "Snacks",
            "category_l2": "Chips & Crisps",
            "storage_temp": "Ambient",
            "shelf_life_days": 180,
            "allergens": ["Wheat", "Soy"],
            "dietary_claims": ["Gluten-Free"],
            "ingredients_summary": "Potatoes, sunflower oil, BBQ seasoning (sugar, salt, paprika, garlic, onion, natural smoke flavor)"
        },
        "_confidence": {
            "category_l1": 0.99,
            "category_l2": 0.95,
            "storage_temp": 0.98,
            "shelf_life_days": 0.70,
            "allergens": 0.55,
            "dietary_claims": 0.40,
            "ingredients_summary": 0.35
        }
    },
    {
        "product": "Lavender Calm Hand Soap 3-Pack",
        "attributes": {
            "category_l1": "Health & Beauty",
            "category_l2": "Hand & Body Care",
            "storage_temp": "Ambient",
            "shelf_life_days": 730,
            "cruelty_free": None,
            "paraben_free": None,
            "key_ingredients": "Lavender essential oil, aloe vera, vitamin E"
        },
        "_confidence": {
            "category_l1": 0.99,
            "category_l2": 0.94,
            "storage_temp": 0.99,
            "shelf_life_days": 0.80,
            "cruelty_free": 0.0,
            "paraben_free": 0.0,
            "key_ingredients": 0.65
        }
    }
]

print("Enrichment results (simulated Bedrock output):")
for result in enrichment_results:
    print(f"\n{'='*50}")
    print(f"Product: {result['product']}")
    print(f"Attributes extracted: {len(result['attributes'])}")
    low_conf = [k for k, v in result['_confidence'].items() if v < 0.60]
    print(f"Low confidence (<0.60): {low_conf if low_conf else 'None'}")

## 4. Validation Layer

Deterministic validation catches errors that foundation models make:
- Enum values that don't match the taxonomy
- Contradictory attributes ("Gluten-Free" with "Wheat" allergen)
- Missing required fields
- Low-confidence attributes that need human review

In [ ]:
CONFIDENCE_THRESHOLD = 0.60
AUTO_PUBLISH_THRESHOLD = 0.85

def validate_enrichment(result, taxonomy):
    """Validate enriched attributes against taxonomy rules."""
    issues = []
    attrs = result['attributes']
    conf = result['_confidence']
    
    # Check category exists in tree
    cat_l1 = attrs.get('category_l1')
    cat_l2 = attrs.get('category_l2')
    if cat_l1 and cat_l1 not in taxonomy['category_tree']:
        issues.append(f"FAIL: category_l1 '{cat_l1}' not in taxonomy")
    elif cat_l1 and cat_l2 and cat_l2 not in taxonomy['category_tree'].get(cat_l1, []):
        issues.append(f"FAIL: category_l2 '{cat_l2}' not under '{cat_l1}'")
    
    # Check storage_temp enum
    valid_temps = ["Ambient", "Refrigerated", "Frozen"]
    if attrs.get('storage_temp') and attrs['storage_temp'] not in valid_temps:
        issues.append(f"FAIL: storage_temp '{attrs['storage_temp']}' invalid")
    
    # Cross-attribute consistency: gluten-free + wheat allergen
    if 'dietary_claims' in attrs and 'allergens' in attrs:
        if 'Gluten-Free' in (attrs.get('dietary_claims') or []):
            if 'Wheat' in (attrs.get('allergens') or []):
                issues.append("CONTRADICTION: 'Gluten-Free' claim with 'Wheat' allergen")
    
    # Check for null required attributes
    for attr_name, value in attrs.items():
        if value is None:
            issues.append(f"MISSING: '{attr_name}' could not be determined")
    
    # Flag low-confidence attributes
    low_conf_attrs = [k for k, v in conf.items() if v < CONFIDENCE_THRESHOLD and v > 0]
    
    # Calculate quality score
    non_null_attrs = [k for k, v in attrs.items() if v is not None]
    completeness = len(non_null_attrs) / len(attrs)
    avg_confidence = sum(v for v in conf.values() if v > 0) / max(len([v for v in conf.values() if v > 0]), 1)
    has_contradictions = any("CONTRADICTION" in i for i in issues)
    
    quality_score = (
        0.4 * completeness +
        0.4 * avg_confidence +
        0.2 * (0.0 if has_contradictions else 1.0)
    )
    
    # Determine routing
    if has_contradictions or any("FAIL" in i for i in issues):
        routing = "REJECT — fix and reprocess"
    elif quality_score >= AUTO_PUBLISH_THRESHOLD:
        routing = "AUTO-PUBLISH to catalog"
    elif quality_score >= CONFIDENCE_THRESHOLD:
        routing = "HUMAN REVIEW queue"
    else:
        routing = "REJECT — insufficient data"
    
    return {
        "product": result['product'],
        "quality_score": round(quality_score, 3),
        "routing": routing,
        "issues": issues,
        "low_confidence_attrs": low_conf_attrs
    }

# Validate all enriched products
print("VALIDATION RESULTS")
print("=" * 60)
for result in enrichment_results:
    validation = validate_enrichment(result, taxonomy)
    print(f"\nProduct: {validation['product']}")
    print(f"  Quality Score: {validation['quality_score']}")
    print(f"  Routing: {validation['routing']}")
    if validation['issues']:
        print(f"  Issues:")
        for issue in validation['issues']:
            print(f"    - {issue}")
    if validation['low_confidence_attrs']:
        print(f"  Low confidence: {validation['low_confidence_attrs']}")

## 5. Multi-Channel Description Generation

With attributes enriched, we generate product descriptions tailored to each output channel. The foundation model receives the enriched product record and channel-specific constraints.

In [ ]:
def build_description_prompt(product, enriched_attrs, channel_spec):
    """Build a description generation prompt for a specific channel."""
    return f"""Write a product description for the following item.

## Product
- Name: {product['product_name']}
- Brand: {product['brand']}
- Size: {product['size']}
- Category: {enriched_attrs.get('category_l1', '')} > {enriched_attrs.get('category_l2', '')}
- Key attributes: {json.dumps(enriched_attrs, default=str)}

## Channel Requirements
- Channel: {channel_spec['channel']}
- Maximum words: {channel_spec['max_words']}
- Tone: {channel_spec['tone']}

## Rules
- Do not make claims not supported by the product attributes.
- Do not use superlatives unless they are factual (e.g., "#1 selling" only if true).
- Include relevant dietary/allergen info for food products.
- Stay within the word limit.

Write only the description text. No headers or labels."""

# Simulated generated descriptions for the oat milk product
descriptions = {
    "ecommerce": (
        "Made from organic oats and filtered water, Nature's Path Organic Oat Milk "
        "delivers a creamy, plant-based alternative that works as hard as dairy. "
        "Pour it over morning cereal, froth it for your afternoon latte, or drink it "
        "straight from the glass. The subtle natural sweetness comes from the oats "
        "themselves — no added sugars, no artificial flavors. Certified organic and "
        "vegan. Shelf-stable until opened, then refrigerate and enjoy within 7-10 days. "
        "32 fl oz carton."
    ),
    "marketplace": (
        "Nature's Path Organic Oat Milk Original 32oz — Creamy plant-based milk made "
        "from certified organic oats. Vegan, no artificial flavors. Froths well for "
        "coffee. Shelf-stable. 12-pack available."
    ),
    "b2b_catalog": (
        "Nature's Path Organic Oat Milk Original. 32 fl oz aseptic carton. "
        "Case pack: 12. Shelf-stable, 270-day shelf life. Certified Organic, Vegan. "
        "Ambient storage. No refrigeration required until opened."
    ),
    "shelf_tag": (
        "Organic oat milk. Vegan. No artificial flavors. 32 fl oz."
    )
}

print("Generated Descriptions — Organic Oat Milk Original")
print("=" * 60)
for channel, desc in descriptions.items():
    word_count = len(desc.split())
    channel_spec = next(c for c in taxonomy['description_channels'] if c['channel'] == channel)
    within_limit = "✓" if word_count <= channel_spec['max_words'] else "✗ OVER LIMIT"
    print(f"\n[{channel.upper()}] ({word_count} words, max {channel_spec['max_words']}) {within_limit}")
    print(f"  {desc}")

## 6. Summary: Pipeline Output

The final enriched product record combines the original supplier data, extracted attributes, validation results, and generated descriptions into a single document ready for PIM import.

In [ ]:
# Assemble final enriched product record
enriched_record = {
    "sku": raw_feed[0]['upc'],
    "supplier_sku": raw_feed[0]['supplier_sku'],
    "brand": raw_feed[0]['brand'],
    "product_name": raw_feed[0]['product_name'],
    "size": raw_feed[0]['size'],
    "case_pack": raw_feed[0]['case_pack'],
    "enriched_attributes": enrichment_results[0]['attributes'],
    "confidence_scores": enrichment_results[0]['_confidence'],
    "descriptions": descriptions,
    "quality_score": 0.763,
    "routing_decision": "HUMAN REVIEW queue",
    "review_reasons": [
        "ingredients_summary confidence below threshold (0.45)",
        "allergens confidence below threshold (0.55) — requires human verification"
    ],
    "enrichment_metadata": {
        "model": "anthropic.claude-sonnet-4-20250514",
        "enrichment_timestamp": "2026-06-22T14:30:00Z",
        "pipeline_version": "1.2.0",
        "processing_time_ms": 2847
    }
}

print("FINAL ENRICHED PRODUCT RECORD")
print("=" * 60)
print(json.dumps(enriched_record, indent=2, default=str))

print(f"\n{'='*60}")
print(f"PIPELINE SUMMARY")
print(f"{'='*60}")
print(f"  Input: 3 raw supplier products (avg 7 fields each)")
print(f"  Output: 3 enriched records (avg 14 fields + 4 descriptions each)")
print(f"  Auto-publishable: 0 of 3 (quality scores below 0.85)")
print(f"  Human review queue: 2 of 3")
print(f"  Rejected (contradictions): 1 of 3")
print(f"  Estimated cost: $0.91 per product × 3 = $2.73")
print(f"  Processing time: ~3 seconds per product (batched)")

## Key Observations

1. **The model is good at categorization and bad at ingredient lists.** Category confidence is consistently above 0.94. Ingredient confidence is below 0.50 because the supplier description does not contain a full ingredient list — the model would need the actual packaging image or spec sheet to extract this reliably.

2. **Validation catches real errors.** The BBQ chips example shows a "Gluten-Free" claim alongside a "Wheat" allergen — a contradiction the model generated because it inferred Gluten-Free from "kettle chips" (potato-based) while also correctly noting that BBQ seasoning often contains wheat-derived ingredients. The validation layer prevents this from reaching the catalog.

3. **Most items route to human review, not auto-publish.** This is expected and correct for initial deployment. As the model's confidence calibration improves through feedback loops (merchandiser edits fed back as training signal), the auto-publish rate rises over months 3-6 from ~15% to 40-60%.

4. **Multi-channel descriptions are the highest-volume inference cost but lowest risk.** A slightly imperfect description costs less than a slightly wrong allergen declaration. The system appropriately applies stricter validation to safety-critical attributes while allowing descriptions more autonomy.

5. **The pipeline is worth running even with 0% auto-publish rate.** A merchandiser reviewing a pre-enriched record (approve/edit/reject) processes 60-80 items per day versus 15-25 from scratch. The value is throughput acceleration, not full automation.